# Logistic Regression Experiment — IEEE-CIS Fraud Detection

## 0. Setup & Imports

In [6]:
from kaggle_secrets import UserSecretsClient
import os
os.environ['DAGSHUB_USER_TOKEN'] = UserSecretsClient().get_secret('DAGSHUB_TOKEN')
print('Token loaded!')

Token loaded!


In [7]:
import subprocess
subprocess.run(['pip', 'install', 'mlflow', 'dagshub', 'optuna', '--quiet'], capture_output=True)

import warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import mlflow, mlflow.sklearn, dagshub, optuna
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.base import BaseEstimator, TransformerMixin
print('Ready!')

Ready!


In [8]:
dagshub.init(repo_owner='ekali-star', repo_name='fraud-detection-ml', mlflow=True)
mlflow.set_experiment('LogisticRegression_Training')
print('Connected:', mlflow.get_tracking_uri())

Initialized MLflow to track repo "ekali-star/fraud-detection-ml"

Repository ekali-star/fraud-detection-ml initialized!

2026/05/08 19:29:03 INFO mlflow.tracking.fluent: Experiment with name 'LogisticRegression_Training' does not exist. Creating a new experiment.


Connected: https://dagshub.com/ekali-star/fraud-detection-ml.mlflow


## 1. Data Loading

In [9]:
BASE = '/kaggle/input/competitions/ieee-fraud-detection/'
train = pd.read_csv(BASE+'train_transaction.csv').merge(pd.read_csv(BASE+'train_identity.csv'), on='TransactionID', how='left')
test  = pd.read_csv(BASE+'test_transaction.csv').merge(pd.read_csv(BASE+'test_identity.csv'),  on='TransactionID', how='left')
print(train.shape, test.shape)

(590540, 434) (506691, 433)


## 2. Cleaning

In [10]:
with mlflow.start_run(run_name='LR_Cleaning'):
    drop_cols = train.isnull().mean()[lambda x: x > 0.5].index.tolist()
    train.drop(columns=drop_cols, inplace=True)
    test.drop(columns=[c for c in drop_cols if c in test.columns], inplace=True)

    const_cols = [c for c in train.columns if train[c].nunique(dropna=False) <= 1]
    train.drop(columns=const_cols, inplace=True)
    test.drop(columns=[c for c in const_cols if c in test.columns], inplace=True)

    for col in ['P_emaildomain', 'R_emaildomain']:
        if col in train.columns:
            top = train[col].value_counts().nlargest(10).index
            train[col] = train[col].where(train[col].isin(top), 'other')
            if col in test.columns:
                test[col] = test[col].where(test[col].isin(top), 'other')

    mlflow.log_param('missing_threshold', 0.5)
    mlflow.log_param('dropped_high_missing', len(drop_cols))
    mlflow.log_metric('cols_remaining', train.shape[1])
    print(f'After cleaning: {train.shape}')

After cleaning: (590540, 220)
🏃 View run LR_Cleaning at: https://dagshub.com/ekali-star/fraud-detection-ml.mlflow/#/experiments/4/runs/03792197c3004d29a50c716c784a549b
🧪 View experiment at: https://dagshub.com/ekali-star/fraud-detection-ml.mlflow/#/experiments/4


## 3. Feature Engineering

In [11]:
with mlflow.start_run(run_name='LR_Feature_Engineering'):

    def engineer(df):
        df = df.copy()
        df['hour']        = (df['TransactionDT'] / 3600) % 24
        df['day_of_week'] = (df['TransactionDT'] / (3600*24)) % 7
        df['is_night']    = ((df['hour'] >= 22) | (df['hour'] <= 6)).astype(int)
        df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)
        df['TransactionAmt_log']   = np.log1p(df['TransactionAmt'])
        df['TransactionAmt_sqrt']  = np.sqrt(df['TransactionAmt'])
        df['TransactionAmt_cents'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)
        if 'P_emaildomain' in df.columns and 'R_emaildomain' in df.columns:
            df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
        df['nan_count'] = df.isnull().sum(axis=1)
        return df

    train = engineer(train)
    test  = engineer(test)

    TARGET    = 'isFraud'
    DROP_COLS = ['TransactionID', 'TransactionDT', TARGET]

    cat_cols = [c for c in train.select_dtypes(include='object').columns if c not in DROP_COLS]
    for col in cat_cols:
        le = LabelEncoder()
        le.fit(train[col].fillna('missing').astype(str))
        train[col] = le.transform(train[col].fillna('missing').astype(str))
        if col in test.columns:
            vals = test[col].fillna('missing').astype(str)
            vals = vals.map(lambda x: x if x in le.classes_ else le.classes_[0])
            test[col] = le.transform(vals)

    feature_cols = [c for c in train.columns if c not in DROP_COLS]
    train_medians = train[feature_cols].median()
    X = train[feature_cols].fillna(train_medians)
    y = train[TARGET]

    X_test = test[[f for f in feature_cols if f in test.columns]].copy()
    for col in feature_cols:
        if col not in X_test.columns:
            X_test[col] = 0
    X_test = X_test[feature_cols].fillna(train_medians)

    mlflow.log_metric('features_after_fe', X.shape[1])
    print(f'Shape: {X.shape}')

Shape: (590540, 225)
🏃 View run LR_Feature_Engineering at: https://dagshub.com/ekali-star/fraud-detection-ml.mlflow/#/experiments/4/runs/da9cdea130cd4fcfb1387eb6987f042b
🧪 View experiment at: https://dagshub.com/ekali-star/fraud-detection-ml.mlflow/#/experiments/4


## 4. Feature Selection

In [ ]:
with mlflow.start_run(run_name='LR_Feature_Selection'):

    # Method 1: SelectKBest
    selector = SelectKBest(f_classif, k=80)
    selector.fit(X, y)
    kbest_features = [feature_cols[i] for i in selector.get_support(indices=True)]

    # Method 2: L1 regularization
    scaler_tmp = StandardScaler()
    X_scaled_tmp = scaler_tmp.fit_transform(X)
    lasso = LogisticRegression(C=0.01, penalty='l1', solver='liblinear', max_iter=1000)
    lasso.fit(X_scaled_tmp, y)
    lasso_mask     = lasso.coef_[0] != 0
    lasso_features = [f for f, m in zip(feature_cols, lasso_mask) if m]

    selected = list(set(kbest_features) | set(lasso_features))

    mlflow.log_param('fs_method_1', 'SelectKBest_f_classif_k80')
    mlflow.log_param('fs_method_2', 'L1_Lasso_C001')
    mlflow.log_metric('kbest_selected', len(kbest_features))
    mlflow.log_metric('lasso_selected', len(lasso_features))
    mlflow.log_metric('union_selected', len(selected))

    X_sel      = X[selected]
    X_test_sel = X_test[selected]
    print(f'KBest: {len(kbest_features)} | Lasso: {len(lasso_features)} | Union: {len(selected)}')

## 5. Training

### 5a. Underfitted — Very Strong Regularization

In [ ]:
with mlflow.start_run(run_name='LR_Underfitted'):
    params_u = dict(C=0.0001, penalty='l2', solver='lbfgs', max_iter=100)
    scaler_u = StandardScaler()
    X_scaled = scaler_u.fit_transform(X_sel)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(LogisticRegression(**params_u), X_scaled, y, cv=cv, scoring='roc_auc')
    mlflow.log_params(params_u)
    mlflow.log_metric('cv_auc_mean', scores.mean())
    mlflow.log_param('note', 'underfitted_very_high_regularization')
    print(f'[UNDERFITTED] CV AUC: {scores.mean():.4f}')

### 5b. Overfitted — Very Low Regularization

In [ ]:
with mlflow.start_run(run_name='LR_Overfitted'):
    params_o = dict(C=1000, penalty='l2', solver='lbfgs', max_iter=5000)
    scaler_o = StandardScaler()
    X_scaled_o = scaler_o.fit_transform(X_sel)
    m_o = LogisticRegression(**params_o)
    m_o.fit(X_scaled_o, y)
    train_auc = roc_auc_score(y, m_o.predict_proba(X_scaled_o)[:, 1])
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_auc = cross_val_score(LogisticRegression(**params_o), X_scaled_o, y,
                              cv=cv, scoring='roc_auc').mean()
    mlflow.log_params(params_o)
    mlflow.log_metric('train_auc', train_auc)
    mlflow.log_metric('cv_auc', cv_auc)
    mlflow.log_metric('overfit_gap', train_auc - cv_auc)
    print(f'[OVERFIT] Train: {train_auc:.4f} CV: {cv_auc:.4f}')

### 5c. Optuna Tuning

In [ ]:
scaler_final = StandardScaler()
X_scaled_final = scaler_final.fit_transform(X_sel)
X_test_scaled  = scaler_final.transform(X_test_sel)

def lr_objective(trial):
    params = {
        'C':        trial.suggest_float('C', 1e-3, 10.0, log=True),
        'penalty':  trial.suggest_categorical('penalty', ['l1', 'l2']),
        'solver':   'liblinear', 'max_iter': 1000
    }
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    return cross_val_score(LogisticRegression(**params), X_scaled_final, y,
                           cv=cv, scoring='roc_auc').mean()

study = optuna.create_study(direction='maximize')
study.optimize(lr_objective, n_trials=15)
best_params = study.best_params
best_params.update({'solver': 'liblinear', 'max_iter': 1000})
print(f'Best AUC: {study.best_value:.4f} | Params: {best_params}')

### 5d. Final CV + Pipeline

In [ ]:
with mlflow.start_run(run_name='LR_Final_CV'):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(y))
    test_preds = np.zeros(len(X_test_scaled))
    fold_aucs  = []

    for fold, (tr_i, val_i) in enumerate(cv.split(X_scaled_final, y)):
        m = LogisticRegression(**best_params)
        m.fit(X_scaled_final[tr_i], y.iloc[tr_i])
        val_pred   = m.predict_proba(X_scaled_final[val_i])[:, 1]
        oof[val_i] = val_pred
        test_preds += m.predict_proba(X_test_scaled)[:, 1] / 5
        fa = roc_auc_score(y.iloc[val_i], val_pred)
        fold_aucs.append(fa)
        print(f'  Fold {fold+1}: {fa:.4f}')

    oof_auc = roc_auc_score(y, oof)
    mlflow.log_params(best_params)
    mlflow.log_metric('oof_auc', oof_auc)
    mlflow.log_metric('cv_auc_mean', np.mean(fold_aucs))
    mlflow.log_metric('cv_auc_std', np.std(fold_aucs))
    print(f'OOF AUC: {oof_auc:.4f}')


class LRPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, selected_features=None):
        self.selected_features = selected_features
        self.label_encoders_   = {}
        self.cat_cols_         = []
        self.medians_          = None

    def fit(self, X, y=None):
        df = self._engineer(X.copy())
        self.cat_cols_ = df.select_dtypes(include='object').columns.tolist()
        for col in self.cat_cols_:
            le = LabelEncoder()
            le.fit(df[col].fillna('missing').astype(str))
            self.label_encoders_[col] = le
        if self.selected_features:
            avail = [f for f in self.selected_features if f in df.columns]
            df = df[avail]
        self.medians_ = df.median()
        return self

    def transform(self, X):
        df = self._engineer(X.copy())
        for col in self.cat_cols_:
            if col in df.columns:
                le = self.label_encoders_[col]
                vals = df[col].fillna('missing').astype(str)
                vals = vals.map(lambda x: x if x in le.classes_ else le.classes_[0])
                df[col] = le.transform(vals)
        if self.selected_features:
            avail = [f for f in self.selected_features if f in df.columns]
            df = df[avail]
        return df.fillna(self.medians_)

    def _engineer(self, df):
        df['hour']        = (df['TransactionDT'] / 3600) % 24
        df['day_of_week'] = (df['TransactionDT'] / (3600*24)) % 7
        df['is_night']    = ((df['hour'] >= 22) | (df['hour'] <= 6)).astype(int)
        df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)
        df['TransactionAmt_log']   = np.log1p(df['TransactionAmt'])
        df['TransactionAmt_sqrt']  = np.sqrt(df['TransactionAmt'])
        df['TransactionAmt_cents'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)
        df['nan_count'] = df.isnull().sum(axis=1)
        if 'P_emaildomain' in df.columns and 'R_emaildomain' in df.columns:
            df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
        return df


X_raw = train.drop(columns=['isFraud', 'TransactionID', 'TransactionDT'], errors='ignore')
y_raw = train['isFraud']

lr_pipeline = Pipeline([
    ('preprocessor', LRPreprocessor(selected_features=selected)),
    ('scaler',       StandardScaler()),
    ('classifier',   LogisticRegression(**best_params))
])
lr_pipeline.fit(X_raw, y_raw)

with mlflow.start_run(run_name='LR_Pipeline_Registry'):
    mlflow.log_metric('oof_auc', oof_auc)
    mlflow.sklearn.log_model(
        sk_model=lr_pipeline,
        artifact_path='lr_fraud_pipeline',
        registered_model_name='LogisticRegression_Fraud_Pipeline'
    )
    print('LR pipeline registered!')

np.save('lr_test_preds.npy', test_preds)
print('Done!')